# Preprocessing and Feature Engineering

This notebook turns the labelled dataframe from the previous notebook into leakage-free, model-ready arrays for both modelling arms: clustering and the autoencoder. Every device is processed on its own, and a pooled case combining all three devices is built alongside it.

Only benign traffic is ever used to fit a scaler, prune features, or fit PCA. Attacks appear only in the test split.

In [1]:
# Standard library
import json
import os
from pathlib import Path

# Move into the repository root so config.py and every relative path inside it resolve correctly
if not Path("config.py").exists():  
    os.chdir("..")

# Third-party
import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Local
import config

# Show every column when previewing a DataFrame, instead of truncating with "..."
pd.set_option("display.max_columns", None)

## Load the labelled data

The dataframe saved by the previous notebook holds every chosen device's traffic, already labelled as benign or a specific attack subtype.

It is loaded here and the random seed is set so every random step below is reproducible.

In [2]:
# Seed every random step in this notebook
np.random.seed(config.RANDOM_SEED)

# Load the labelled dataframe saved by the previous notebook
df = pd.read_parquet(Path(config.PROCESSED_DIRECTORY) / "df.parquet")
feature_columns = config.feature_columns(df)

print(f"Rows: {len(df)}")
print(f"Feature columns: {len(feature_columns)}")
print(f"Total columns: {df.shape[1]}")

Rows: 419531
Feature columns: 115
Total columns: 118


## Create the output folders

Processed arrays and fitted transforms for each chosen device are stored in their respective folders. This includes the pooled case which is stored in the folder named `_pooled`.

In [3]:
# Create the output folder for every chosen device and the pooled case
entries = config.CHOSEN_DEVICES + ["_pooled"]
for entry in entries:
    (Path(config.PROCESSED_DIRECTORY) / entry).mkdir(parents=True, exist_ok=True)
    (Path(config.MODEL_DIRECTORY) / entry).mkdir(parents=True, exist_ok=True)

## Split each device into benign-only train/validation and a mixed test set

For each device, the benign rows are split 70/15/15 into train, validation, and test. Every attack row is added only to that device's test split, so train and validation never contain an attack, and all later benign-only fitting stays leakage-free.

In [4]:
# Split every device's rows into a benign-only train/validation and a mixed test set
device_splits = {}
for device in config.CHOSEN_DEVICES:
    device_dataframe = df[df["device"] == device]
    train, val, test = config.split_benign_only(device_dataframe)
    device_splits[device] = {"train": train, "val": val, "test": test}

    # Train and validation must never contain an attack row
    assert set(train["label"].unique()) == {"benign"}
    assert set(val["label"].unique()) == {"benign"}

    # Train, validate, and the benign share of test must partition the benign rows exactly, with no row dropped or counted twice by the split
    benign_total = (device_dataframe["label"] == "benign").sum()
    benign_in_test = (test["label"] == "benign").sum()
    assert len(train) + len(val) + benign_in_test == benign_total

    attack_rows_in_test = (test["label"] != "benign").sum()
    print(f"{device}: Train: {len(train)}, Validate: {len(val)}, Test: {len(test)} (Benign: {benign_in_test}, Attack: {attack_rows_in_test})")

Danmini_Doorbell: Train: 28276, Validate: 6059, Test: 46060 (Benign: 6060, Attack: 40000)
Philips_B120N10_Baby_Monitor: Train: 115598, Validate: 24771, Test: 64772 (Benign: 24772, Attack: 40000)
Provision_PT_838_Security_Camera: Train: 65796, Validate: 14099, Test: 54100 (Benign: 14100, Attack: 40000)


## Prune correlated features, then standardise

Two things happen here, in this order, both fitted on benign train only.

First, redundant features are identified: any feature that never changes across the benign training rows is set aside just for this step (a constant feature has no variance, so its correlation with anything is undefined), and then, of every remaining pair of features correlated above the threshold, the one with lower variance is dropped. This runs on the raw, unscaled data, because after standardising every feature would have the same variance and the choice of which one to drop would become arbitrary.

Second, a `StandardScaler` is fitted on every feature column (constants included, so the saved scaler always expects the same columns) and used to transform train, validation, and test. The autoencoder's input is then this scaled data, restricted to the surviving, uncorrelated columns.

The count of constant columns found is printed for each device; a feature that is constant for benign traffic but not for attacks would be a useful signal that this step discards, so it is worth checking that count stays low.

In [5]:
# Prune correlated features (on raw benign train) and standardise (on full benign train). This is shared by the per-device loop below and the pooled case later,
# so both stay identical
def prune_and_standardise(train, val, feature_columns):
    # A constant feature has zero variance in benign train, so its correlation with anything is undefined. Constant columns are set it aside before pruning
    variance = train[feature_columns].var()
    constant_columns = variance[variance == 0].index.tolist()
    non_constant_columns = [column for column in feature_columns if column not in constant_columns]

    # Drop the lower-variance column of each correlated pair, computed on the non-constant, unscaled benign train data
    kept_columns = config.prune_correlated_columns(train, non_constant_columns)

    # Fit the scaler on every feature column in the training set, constants included, so its column contract stays identical everywhere it is reused
    standard_scaler = StandardScaler().fit(train[feature_columns])

    # Transform train and val with that same fitted scaler. test is transformed later, once it exists, using the returned scaler and kept_columns
    X_train_autoencoder = transform_autoencoder(train, standard_scaler, kept_columns, feature_columns)
    X_val_autoencoder = transform_autoencoder(val, standard_scaler, kept_columns, feature_columns)

    return (X_train_autoencoder, X_val_autoencoder, standard_scaler, kept_columns, len(constant_columns))


# Scale a dataframe with an already-fitted scaler, then restrict to kept_columns. This is used for train/val above, and again for test once it exists.
def transform_autoencoder(dataframe, standard_scaler, kept_columns, feature_columns):
    # Scale every feature column first, keeping the scaler's full column width
    scaled = pd.DataFrame(standard_scaler.transform(dataframe[feature_columns]), columns=feature_columns)
    # Restrict to the pruned columns only for the autoencoder's actual input
    return scaled[kept_columns].to_numpy()

In [6]:
# Apply the pruning and standardising step to each device in turn
device_autoencoder = {}
device_kept_columns = {}
device_constant_counts = {}
for device, splits in device_splits.items():
    # Fit pruning and the scaler on this device's own benign train/val only
    X_train_autoencoder, X_val_autoencoder, standard_scaler, kept_columns, constant_column_count = (prune_and_standardise(splits["train"], splits["val"], feature_columns))
    # Transform this device's test split with what was just fitted. test itself is never used to fit anything
    X_test_autoencoder = transform_autoencoder(splits["test"], standard_scaler, kept_columns, feature_columns)

    # Keep all three representations together, ready for the PCA step that follows
    device_autoencoder[device] = {"train": X_train_autoencoder, "val": X_val_autoencoder, "test": X_test_autoencoder}
    device_kept_columns[device] = kept_columns
    device_constant_counts[device] = constant_column_count

    # Save this device's fitted scaler and surviving column list to disk
    joblib.dump(standard_scaler, Path(config.MODEL_DIRECTORY) / device / "scaler.joblib")
    with open(Path(config.MODEL_DIRECTORY) / device / "kept_columns.json", "w") as kept_columns_file:
        json.dump(kept_columns, kept_columns_file)

    print(f"{device}: Constant columns: {constant_column_count}, Columns kept: {len(kept_columns)}")

Danmini_Doorbell: Constant columns: 0, Columns kept: 52
Philips_B120N10_Baby_Monitor: Constant columns: 0, Columns kept: 52
Provision_PT_838_Security_Camera: Constant columns: 0, Columns kept: 41


## Reduce dimensionality with PCA for the clustering representation

Distance-based clustering (K-Means and DBSCAN) works better in a lower-dimensional space. Hence, PCA is fitted on the benign training autoencoder representation to keep enough components to explain the configured share of variance, and used to transform all three splits.

In [7]:
# Fit PCA on the benign training autoencoder representation, per device
device_clustering = {}
for device, arrays in device_autoencoder.items():
    # Fit PCA on this device's own benign train only to keep enough components to explain the configured share of variance
    pca = PCA(n_components=config.PCA_VARIANCE, random_state=config.RANDOM_SEED, svd_solver="full").fit(arrays["train"])

    # Transform train, val, and test with that same fitted PCA
    device_clustering[device] = {"train": pca.transform(arrays["train"]), "val": pca.transform(arrays["val"]), "test": pca.transform(arrays["test"])}
    # Save this device's fitted PCA transform to disk
    joblib.dump(pca, Path(config.MODEL_DIRECTORY) / device / "pca.joblib")

    print(f"{device}: retained components={pca.n_components_}")

Danmini_Doorbell: retained components=22
Philips_B120N10_Baby_Monitor: retained components=19
Provision_PT_838_Security_Camera: retained components=15


## Build the test labels and metadata

The label is turned into a simple binary target for evaluation (1 for any attack, 0 for benign). The attack subtype is kept alongside it so that results can later be broken down by attack type.

Both are taken from the untouched test split, so their row order matches the saved arrays exactly. This is only used for evaluation in a later notebook.

In [8]:
# Build the evaluation-only label and metadata for each device's test split
device_y_test = {}
device_test_metadata = {}
for device, splits in device_splits.items():
    test = splits["test"]
    device_y_test[device] = (test["label"] != "benign").astype(int).to_numpy()
    device_test_metadata[device] = test[["device", "attack_subtype"]].reset_index(drop=True)

## Save each device's arrays

Every array built so far for a device is written to that device's folder under `data/processed/`. It is ready for subsequent notebooks to load without needing to recompute anything.

In [9]:
# Save every device's arrays and test metadata to its processed folder
for device in config.CHOSEN_DEVICES:
    config.save_arrays(
        device,
        X_train_autoencoder=device_autoencoder[device]["train"],
        X_val_autoencoder=device_autoencoder[device]["val"],
        X_test_autoencoder=device_autoencoder[device]["test"],
        X_train_clustering=device_clustering[device]["train"],
        X_val_clustering=device_clustering[device]["val"],
        X_test_clustering=device_clustering[device]["test"],
        y_test=device_y_test[device],
    )
    device_test_metadata[device].to_parquet(Path(config.PROCESSED_DIRECTORY) / device / "test_metadata.parquet")

## Build the pooled benign training and validation sets

The pooled model is meant to represent all three devices equally, so its benign train and validation sets each draw the same number of rows from every device: the smallest contributing device's row count, capped at `POOLED_BENIGN_CAP` only if that is unexpectedly large. Without this, the device with the most benign traffic would dominate the pooled statistics and the single shared anomaly threshold set later.

In [10]:
# Pool each device's benign train and validation, balanced to an equal row count per device
pooled_train_raw = pd.concat([device_splits[device]["train"] for device in config.CHOSEN_DEVICES], ignore_index=True)
pooled_val_raw = pd.concat([device_splits[device]["val"] for device in config.CHOSEN_DEVICES], ignore_index=True)

pooled_train = config.balance_rows_per_device(pooled_train_raw)
pooled_val = config.balance_rows_per_device(pooled_val_raw)

print("Pooled train rows per device:")
print(pooled_train["device"].value_counts())
print()
print("Pooled validation rows per device:")
print(pooled_val["device"].value_counts())

Pooled train rows per device:
device
Danmini_Doorbell                    28276
Philips_B120N10_Baby_Monitor        28276
Provision_PT_838_Security_Camera    28276
Name: count, dtype: int64

Pooled validation rows per device:
device
Danmini_Doorbell                    6059
Philips_B120N10_Baby_Monitor        6059
Provision_PT_838_Security_Camera    6059
Name: count, dtype: int64


## Fit one shared scaler, prune, and PCA for the pooled case

The same pruning-then-standardising step used per device above is used again on the pooled benign train and validation built just now. This is done to produce one shared scaler, one shared set of kept columns, and one shared PCA transform for the pooled model.

In [11]:
# Prune, standardise, and reduce dimensionality once, for the pooled benign train/val
X_pooled_train_autoencoder, X_pooled_val_autoencoder, pooled_scaler, pooled_kept_columns, pooled_constant_count = (prune_and_standardise(pooled_train, pooled_val, feature_columns))

pooled_pca = PCA(n_components=config.PCA_VARIANCE, random_state=config.RANDOM_SEED, svd_solver="full").fit(X_pooled_train_autoencoder)

X_pooled_train_clustering = pooled_pca.transform(X_pooled_train_autoencoder)
X_pooled_val_clustering = pooled_pca.transform(X_pooled_val_autoencoder)

joblib.dump(pooled_scaler, Path(config.MODEL_DIRECTORY) / "_pooled" / "scaler.joblib")
joblib.dump(pooled_pca, Path(config.MODEL_DIRECTORY) / "_pooled" / "pca.joblib")
with open(Path(config.MODEL_DIRECTORY) / "_pooled" / "kept_columns.json", "w") as kept_columns_file:
    json.dump(pooled_kept_columns, kept_columns_file)

print(f"_pooled: Constant columns: {pooled_constant_count}, Columns kept: {len(pooled_kept_columns)}, Retained components: {pooled_pca.n_components_}")

_pooled: Constant columns: 0, Columns kept: 53, Retained components: 19


## Build the pooled test set from each device's own test rows

The pooled test set is not a new split. It is each device's already-built test split (which was never touched by any device's own scaler or PCA fitting) concatenated together, then transformed with the pooled scaler, kept columns, and PCA.

No row is ever re-split or moved between devices, so no device's benign or attack traffic can leak into another device's train or validation.

In [12]:
# Concatenate each device's own test rows, unchanged, then transform with the pooled scaler/PCA
pooled_test = pd.concat([device_splits[device]["test"] for device in config.CHOSEN_DEVICES], ignore_index=True)

X_pooled_test_autoencoder = transform_autoencoder(pooled_test, pooled_scaler, pooled_kept_columns, feature_columns)
X_pooled_test_clustering = pooled_pca.transform(X_pooled_test_autoencoder)

y_pooled_test = (pooled_test["label"] != "benign").astype(int).to_numpy()
pooled_test_metadata = pooled_test[["device", "attack_subtype"]].reset_index(drop=True)

print(f"Pooled test rows: {len(pooled_test):,}")
print(pooled_test_metadata["device"].value_counts())

Pooled test rows: 164,932
device
Philips_B120N10_Baby_Monitor        64772
Provision_PT_838_Security_Camera    54100
Danmini_Doorbell                    46060
Name: count, dtype: int64


## Save the pooled arrays

The pooled arrays are saved in exactly the same layout as a device's own folder, under `_pooled`, so later notebooks can loop over `config.CHOSEN_DEVICES + ["_pooled"]` without any special-casing.

In [13]:
# Save the pooled arrays and test metadata alongside the per-device folders
config.save_arrays(
    "_pooled",
    X_train_autoencoder=X_pooled_train_autoencoder,
    X_val_autoencoder=X_pooled_val_autoencoder,
    X_test_autoencoder=X_pooled_test_autoencoder,
    X_train_clustering=X_pooled_train_clustering,
    X_val_clustering=X_pooled_val_clustering,
    X_test_clustering=X_pooled_test_clustering,
    y_test=y_pooled_test,
)
pooled_test_metadata.to_parquet(Path(config.PROCESSED_DIRECTORY) / "_pooled" / "test_metadata.parquet")

## Verify the saved outputs

Everything just written is reloaded from disk and checked. Row counts must agree across the two representations of the same split, the column counts saved alongside each array must match what the fitted transforms actually produce, and the scaler must still expect every original feature column, not a constant-dropped subset.

In [14]:
# Reload every saved file per entry and check shapes, column counts, and the scaler's contract
entries = config.CHOSEN_DEVICES + ["_pooled"]
constant_counts = {**device_constant_counts, "_pooled": pooled_constant_count}
reloaded_kept_columns = {}
reloaded_test_metadata = {}
reloaded_y_test = {}
summary_rows = []
for entry in entries:
    # Build this entry's processed/model folder paths
    processed_directory = Path(config.PROCESSED_DIRECTORY) / entry
    model_directory = Path(config.MODEL_DIRECTORY) / entry

    # Reload every saved array and the test metadata from disk
    X_train_autoencoder = np.load(processed_directory / "X_train_autoencoder.npy")
    X_val_autoencoder = np.load(processed_directory / "X_val_autoencoder.npy")
    X_test_autoencoder = np.load(processed_directory / "X_test_autoencoder.npy")
    X_train_clustering = np.load(processed_directory / "X_train_clustering.npy")
    X_val_clustering = np.load(processed_directory / "X_val_clustering.npy")
    X_test_clustering = np.load(processed_directory / "X_test_clustering.npy")
    y_test = np.load(processed_directory / "y_test.npy")
    test_metadata = pd.read_parquet(processed_directory / "test_metadata.parquet")
    reloaded_test_metadata[entry] = test_metadata
    reloaded_y_test[entry] = y_test

    # Reload the fitted scaler, PCA, and the surviving column list from disk
    standard_scaler = joblib.load(model_directory / "scaler.joblib")
    pca = joblib.load(model_directory / "pca.joblib")
    with open(model_directory / "kept_columns.json") as kept_columns_file:
        kept_columns = json.load(kept_columns_file)
    reloaded_kept_columns[entry] = kept_columns

    # Row counts must agree across both representations of the same split
    assert (X_test_autoencoder.shape[0] == X_test_clustering.shape[0] == y_test.shape[0] == len(test_metadata))
    assert X_train_autoencoder.shape[0] == X_train_clustering.shape[0]
    assert X_val_autoencoder.shape[0] == X_val_clustering.shape[0]

    # Column counts must match what the fitted transforms actually produce
    assert len(kept_columns) == X_train_autoencoder.shape[1]
    assert pca.n_components_ == X_train_clustering.shape[1]
    assert list(standard_scaler.feature_names_in_) == feature_columns

    # Record this entry's counts for the summary table below
    summary_rows.append({
        "entry": entry,
        "train_rows": X_train_autoencoder.shape[0],
        "val_rows": X_val_autoencoder.shape[0],
        "test_rows": X_test_autoencoder.shape[0],
        "constant_columns": constant_counts[entry],
        "kept_columns": len(kept_columns),
        "pca_components": pca.n_components_
    })

pd.DataFrame(summary_rows)

,entry,train_rows,val_rows,test_rows,constant_columns,kept_columns,pca_components
0,Danmini_Doorbell,28276,6059,46060,0,52,22
1,Philips_B120N10_Baby_Monitor,115598,24771,64772,0,52,19
2,Provision_PT_838_Security_Camera,65796,14099,54100,0,41,15
3,_pooled,84828,18177,164932,0,53,19


## Check whether the retained feature set is the same everywhere

Each device (and the pooled case) prunes correlated features independently, so they are not guaranteed to keep the same columns.

In [15]:
# Report whether every entry kept the same columns or different ones
kept_column_sets = {entry: set(columns) for entry, columns in reloaded_kept_columns.items()}
distinct_sets = {frozenset(columns) for columns in kept_column_sets.values()}

if len(distinct_sets) == 1:
    print(f"Every entry kept the same {len(reloaded_kept_columns[entries[0]])} columns.")
else:
    print("Kept columns differ by entry:")
    for entry, columns in reloaded_kept_columns.items():
        print(f"- {entry}: {len(columns)} columns")

Kept columns differ by entry:
- Danmini_Doorbell: 52 columns
- Philips_B120N10_Baby_Monitor: 52 columns
- Provision_PT_838_Security_Camera: 41 columns
- _pooled: 53 columns


## Check the pooled balancing and the pooled test union

Three things specific to the pooled case are checked here:
- The pooled scaler was fit on an equal number of training rows from every device.
- The pooled validation set is balanced the same way (a flat cap alone does not guarantee this, since every device's validation split is already well under the cap).
- The pooled test set is exactly the union of each device's own test rows, with no row lost, duplicated, or moved to a different device.

In [16]:
# The pooled scaler must have seen an equal share of benign rows from every device
per_device_train_count = min(len(device_splits[device]["train"]) for device in config.CHOSEN_DEVICES)
per_device_train_count = min(per_device_train_count, config.POOLED_BENIGN_CAP)
assert np.all(pooled_scaler.n_samples_seen_ == per_device_train_count * len(config.CHOSEN_DEVICES))

# Pooled validation must be balanced too, not just capped
pooled_val_counts = pooled_val["device"].value_counts()
assert pooled_val_counts.nunique() == 1

# The pooled test set must be a clean union: same row count and same
# benign/attack split as concatenating each device's own test set directly
per_device_test_lengths = [len(device_splits[device]["test"]) for device in config.CHOSEN_DEVICES]
per_device_attack_counts = [device_y_test[device].sum() for device in config.CHOSEN_DEVICES]
assert sum(per_device_test_lengths) == len(y_pooled_test)
assert sum(per_device_attack_counts) == y_pooled_test.sum()

pooled_test_device_counts = reloaded_test_metadata["_pooled"]["device"].value_counts()
for device in config.CHOSEN_DEVICES:
    assert pooled_test_device_counts[device] == len(reloaded_test_metadata[device])

print("Pooled train/val rows per device:", per_device_train_count)
print(pooled_val_counts)
print()
print("Pooled test rows per device (from saved metadata):")
print(pooled_test_device_counts)

Pooled train/val rows per device: 28276
device
Danmini_Doorbell                    6059
Philips_B120N10_Baby_Monitor        6059
Provision_PT_838_Security_Camera    6059
Name: count, dtype: int64

Pooled test rows per device (from saved metadata):
device
Philips_B120N10_Baby_Monitor        64772
Provision_PT_838_Security_Camera    54100
Danmini_Doorbell                    46060
Name: count, dtype: int64


## Check that `kept_columns.json` is reproducible

Each saved `kept_columns.json` should match what pruning produces if recomputed from scratch on the same benign train data.

This is a regression check: if it ever fails, the saved file and the pruning logic have drifted apart.

In [17]:
# Recompute kept_columns from scratch and check it matches the saved JSON exactly
for entry in entries:
    # Use this entry's own benign train dataframe, per device or pooled
    train = pooled_train if entry == "_pooled" else device_splits[entry]["train"]

    # Repeat the same unscaled, constant-excluded pruning used at fit time
    variance = train[feature_columns].var()
    non_constant_columns = variance[variance > 0].index.tolist()
    recomputed_kept_columns = config.prune_correlated_columns(train, non_constant_columns)

    # The freshly recomputed result must match what was saved to disk
    assert recomputed_kept_columns == reloaded_kept_columns[entry]

print("kept_columns.json matches a freshly recomputed pruning for every entry.")

kept_columns.json matches a freshly recomputed pruning for every entry.


## Confirm every expected file exists

A check is done to ensure that all eleven expected files (seven arrays, the test metadata, and three fitted transforms) are actually present on disk.

In [18]:
# Sweep every expected file for every entry and confirm it exists
processed_file_names = ["X_train_autoencoder.npy", "X_val_autoencoder.npy", "X_test_autoencoder.npy", "X_train_clustering.npy", "X_val_clustering.npy", "X_test_clustering.npy", "y_test.npy", "test_metadata.parquet"]
model_file_names = ["scaler.joblib", "pca.joblib", "kept_columns.json"]

missing_files = []
for entry in entries:
    # Check every processed array/metadata file for this entry
    for file_name in processed_file_names:
        if not (Path(config.PROCESSED_DIRECTORY) / entry / file_name).exists():
            missing_files.append(f"data/processed/{entry}/{file_name}")
    # Check every fitted model file for this entry
    for file_name in model_file_names:
        if not (Path(config.MODEL_DIRECTORY) / entry / file_name).exists():
            missing_files.append(f"models/{entry}/{file_name}")

assert not missing_files, f"Missing files: {missing_files}"
print(f"All {len(entries) * (len(processed_file_names) + len(model_file_names))} expected files are present.")

All 44 expected files are present.
